# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [6]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [7]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Loan Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [8]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/complaints.csv",
    metadata_columns=[
      "Date received", 
      "Product", 
      "Sub-product", 
      "Issue", 
      "Sub-issue", 
      "Consumer complaint narrative", 
      "Company public response", 
      "Company", 
      "State", 
      "ZIP code", 
      "Tags", 
      "Consumer consent provided?", 
      "Submitted via", 
      "Date sent to company", 
      "Company response to consumer", 
      "Timely response?", 
      "Consumer disputed?", 
      "Complaint ID"
    ]
)

loan_complaint_data = loader.load()

for doc in loan_complaint_data:
    doc.page_content = doc.metadata["Consumer complaint narrative"]

Let's look at an example document to see if everything worked as expected!

In [9]:
loan_complaint_data[0]

Document(metadata={'source': './data/complaints.csv', 'row': 0, 'Date received': '03/27/25', 'Product': 'Student loan', 'Sub-product': 'Federal student loan servicing', 'Issue': 'Dealing with your lender or servicer', 'Sub-issue': 'Trouble with how payments are being handled', 'Consumer complaint narrative': "The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who are not in the same position. The re-amortization should have occurred once the forbearance ended to reduce the impact to borrowers.", 'Company public response': 'None', 'Company'

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "LoanComplaints".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [10]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    loan_complaint_data,
    embeddings,
    location=":memory:",
    collection_name="LoanComplaints"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [11]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [12]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [13]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [14]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [15]:
naive_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided complaints, appears to be dealing with the lender or servicer, including errors in loan balances, misapplied payments, wrongful denials of payment plans, and issues related to loan mismanagement. Many complaints also involve incorrect or outdated information on credit reports, unauthorized transfers of loans, and difficulties in understanding or accessing accurate loan data.'

In [16]:
naive_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, yes, some complaints did not get handled in a timely manner. Specifically, the complaint associated with row 441 (submitted to MOHELA on 03/28/25) was marked as "No" for timely response, indicating it was not addressed promptly. Additionally, multiple complaints mention delays or failure to respond within expected timeframes, such as the complaint with ID 12832400 (submitted to Maximus Federal Services on 04/05/25), which was marked as "Yes" for timely response, but other complaints, like the one with ID 12709087 (submitted to MOHELA on 03/28/25), explicitly state the response was not timely.\n\nTherefore, the answer is: Yes, some complaints did not get handled in a timely manner.'

In [17]:
naive_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to a combination of factors such as:\n\n1. **Accumulation of interest during forbearance or deferment periods:** Many borrowers were only offered options like forbearance or deferment, during which interest continued to accrue, making it difficult to pay off the principal amount later.\n\n2. **Lack of clear communication and notification:** Several complaints indicate that borrowers were not adequately informed about loan transfers, repayment resumption dates, or changes in payment obligations, leading to unintentional delinquencies.\n\n3. **Financial hardships and stagnant wages:** Borrowers often faced financial difficulties, making it impossible to afford increased payments or even the basic cost of living, especially when their income did not keep pace with rising interest or loan balances.\n\n4. **Mismanagement and errors by loan servicers:** Complaints include issues like incorrect reporting of late payments, inability to apply

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [18]:
from langchain_community.retrievers import BM25Retriever


bm25_retriever = BM25Retriever.from_documents(loan_complaint_data, )

We'll construct the same chain - only changing the retriever.

In [19]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [20]:
bm25_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with loans appears to be problems related to dealing with lenders or servicers, such as incorrect or bad information about loans, issues with repayment terms and application of payments, and disputes over fees or loan validity. Specifically, complaints frequently involve mismanagement of payments, inaccurate loan information, and lack of clarity or transparency from loan servicers.'

In [21]:
bm25_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, all of the complaints mentioned indicate that they were responded to in a timely manner. Specifically, the complaints with IDs 13197090, 12792958, 13160766, and 13410623 all have responses marked as "Yes" for "Timely response." Therefore, there is no evidence in this data that any complaints were not handled in a timely manner.'

In [22]:
bm25_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People often fail to pay back their loans due to a variety of issues, including miscommunication or lack of communication from the loan servicers, problems with payment processing, and complications arising from changes in loan servicing or payment plans. For example, some individuals experienced their autopayments being discontinued without proper notification, leading to missed payments and negative impacts on their credit scores. Others faced difficulties because loan servicers transferred their loans without informing them, or because they were steered into incorrect payment plans or forbearances, which caused interest to accrue and increased their total debt. Additionally, delays or lack of response from loan servicers regarding requests for deferment or forbearance can result in unpaid bills and financial hardship. Overall, failures in communication, administrative errors, and inadequate support from loan servicing companies contribute to borrowers being unable to repay their lo

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

✅ Answer:

Query: “List the symptoms of dengue fever.”

BM25 is better than embeddings for this query because it focuses on exact keyword matching, which is ideal when the query uses domain-specific terms like “symptoms” and “dengue fever.” These terms are likely to appear verbatim in medical texts or lists, enabling BM25 to retrieve documents that directly mention them.

In contrast, embeddings may introduce semantic fuzziness, potentially retrieving documents about similar diseases (e.g., malaria or chikungunya) due to their semantic closeness. For precise, fact-based queries with high keyword specificity, BM25 offers more accurate retrieval.

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [23]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [24]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [25]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided complaints and context, a common issue with loans, especially student loans, appears to be problems related to mismanagement and incorrect information. Specifically, issues such as errors in loan balances, misapplied payments, wrongful denials of payment plans, and mishandling of loan data are prevalent. Additionally, many complaints involve lack of communication, incorrect or inconsistent loan information, and violations of privacy laws. \n\nTherefore, the most common issue with loans seems to be **errors and mismanagement of loan information and account handling**.'

In [26]:
contextual_compression_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, it appears that some complaints did not get handled in a timely manner. For example, one complaint regarding a request for a loan account review has been open for over 1 year with no resolution, and another issue related to loan information release violations has been unresolved for nearly 18 months. Additionally, there are complaints about payments not being applied to accounts that may still be pending resolution. \n\nTherefore, yes, there are complaints that did not get handled in a timely manner.'

In [27]:
contextual_compression_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for several reasons, including:\n\n1. Lack of Awareness and Information: Borrowers were often unaware that they needed to repay their loans or were not properly informed by financial aid officers about repayment obligations. For example, some borrowers did not realize they had to pay back their financial aid until much later.\n\n2. Poor Communication and Notification: Borrowers reported not receiving adequate notifications about payment due dates, account transfers, or changes in loan servicers. This lack of communication led to missed payments and confusion.\n\n3. Difficulties with Payment Handling: Some borrowers experienced trouble with how payments were being processed, including being locked out of online accounts, incorrect account information, or being unaware of when payments were due.\n\n4. Accumulation of Interest and Unmanageable Loan Terms: Borrowers faced ongoing interest accrual, especially during forbearance or deferment periods, wh

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [28]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [29]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [30]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

"Based on the provided context, the most common issues with student loans include:\n\n- Dealing with your lender or servicer, such as trouble with payment handling, applying extra funds, or loan management.\n- Problems with loan balances, interest calculations, or incorrect reporting.\n- Issues with loan documentation, including missing signed promissory notes or legal records.\n- Problems with loan forgiveness, cancellation, or discharge.\n- Challenges in managing repayment plans, including being steered into forbearance with accruing interest.\n- Unauthorized transfers or mismanagement of loans.\n- Poor customer service, lack of communication, or unhelpful representatives.\n- Discrepancies in account status, late payments, or credit reporting errors.\n\nOverall, a recurring theme is that borrowers often face difficulties related to mismanagement, lack of transparency, or improper handling of their loans by servicers and agencies.\n\nIf I had to identify the single most common issue, 

In [31]:
multi_query_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the information provided, yes, some complaints did not get handled in a timely manner. Specifically, there are multiple instances where the responses from the companies were either late or the complaints remain unresolved after extended periods. For example:\n\n- Complaint ID 12739706 (MOHELA in NJ): Response was late, and the complaint was not addressed within the expected timeframe ("Timely response?": "No").\n- Complaint ID 12668396 (MOHELA in NJ): The response was late, and the complaint was not handled timely.\n- Complaint ID 13056764 (EdFinancial in IN): The response was timely.\n- Complaint ID 13157020 (Maximus/Aidvantage in WV): The response was timely.\n- Several other complaints mention ongoing issues despite repeated follow-ups, delays, or lack of resolution over months or even years.\n\nTherefore, yes, some complaints did not get handled in a timely manner.'

In [32]:
multi_query_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People often failed to pay back their loans due to a variety of systemic and individual challenges, including:\n\n1. Lack of clear information about repayment options and interest accumulation — borrowers were often not adequately informed about how interest compounds, the availability of income-driven repayment plans, or forgiveness programs, leading to unmanageable debt growth.\n2. Reliance on forbearance or deferment as the only options, which allowed interest to continue accruing, sometimes causing balances to balloon beyond the original amounts borrowed.\n3. Mismanagement and miscommunication by loan servicers, including inadequate notices about repayment obligations, transfers of loans without proper notification, and incorrect or inconsistent account information.\n4. Financial hardships such as unemployment, medical issues, or unexpected expenses that made consistent repayment difficult or impossible.\n5. Predatory lending practices and loans taken out under false pretenses or 

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

✅ Answer:

Generating multiple reformulations of a user query improves recall by increasing the chances of retrieving relevant documents that use different phrasings or terminology than the original query.

For example, a user may search:

“How can I treat high blood pressure?”

Documents might instead use:
	•	“Managing hypertension”
	•	“Blood pressure control methods”
	•	“Therapies for elevated blood pressure”

If only the original query is used, many relevant documents may be missed due to lack of keyword overlap. By generating multiple reformulations that capture different ways of expressing the same intent, the retrieval system can cast a wider net, ensuring more relevant documents are retrieved, thereby boosting recall.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [40]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = loan_complaint_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [41]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [42]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [43]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [44]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [45]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided complaints, appears to be related to misconduct by loan servicers, including errors in loan balances, misapplied payments, wrongful denials of payment plans, and issues with illegal or unverified credit reporting. Many complaints highlight systemic problems such as incorrect information on credit reports, unfair interest rate increases, and failure to verify the legitimacy of debts.'

In [46]:
parent_document_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, all the complaints documented were marked as "No" under the "Timely response?" field, indicating that they were not handled in a timely manner. Specifically, at least two complaints regarding student loans serviced by MOHELA received responses labeled as "No," meaning the responses were late. Therefore, yes, some complaints did not get handled in a timely manner.'

In [47]:
parent_document_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People often fail to pay back their loans due to various reasons. Based on the provided context, some common reasons include:\n\n1. Financial Hardship: Borrowers experience severe financial difficulties after graduation, making it challenging to make consistent loan payments.\n2. Lack of Proper Information or Communication: Some borrowers are not properly informed about payment schedules, grace periods, or changes in loan servicing, leading to missed or late payments.\n3. Misrepresentation and Lack of Transparency: Borrowers may have taken out loans based on misleading information about the value of their education, career prospects, or loan terms.\n4. Institutional Failures and School Closures: The closure of educational institutions and undisclosed financial problems can result in borrowers being unable to secure employment or repay loans.\n5. Administrative Issues: Errors such as unverified or questionable debt reporting, failure to notify borrowers about payment obligations, or mi

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [48]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [49]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [50]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided data, the most common issues with student loans appear to be related to:\n\n- Dealing with lenders or servicers, including receiving bad information about loans, mishandling of accounts, and improper transfer or reporting of loan status.\n- Problems with payment handling, such as inability to apply payments correctly, incorrect interest calculations, or unauthorized changes to loan balances.\n- Incorrect or misleading information reported to credit bureaus, leading to drops in credit scores and damaging credit history.\n- Lack of communication or notification from loan servicers about account status, transfers, or default issues.\n- Disputes over loan balances, interest calculations, or the legitimacy of the debt, often involving allegations of mismanagement, fraud, or improper validation.\n\nOverall, issues with loan servicing—particularly errors in account management, incorrect reporting, and lack of proper communication—are the most prevalent according to thes

In [51]:
ensemble_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Yes, based on the provided complaints, some complaints indicate that complaints were not handled in a timely manner. For example:\n\n- Complaint ID 12744910 from Maximus Federal Services, Inc. (MI) received on 03/31/25 was marked "Timely response?": Yes, but the response was "Closed with explanation," suggesting delays or unresolved issues.\n- Complaint ID 12935889 from MOHELA (CO) received on 04/11/25 was marked "Timely response?": No, indicating it was not handled promptly.\n- Complaint ID 12739706 from MOHELA (NJ) received on 04/01/25 was marked "Timely response?": No, and the response was "Closed with explanation."\n- Several other complaints, especially those marked "Closed with explanation" or with long wait times and unresolved issues, suggest delays or failures to handle complaints promptly.\n\nOverall, multiple complaints reflect that some issues were not addressed in a timely manner, either due to delays, lack of response, or being marked as closed without resolution.'

In [52]:
ensemble_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for several reasons evident from the complaints:\n\n1. **Lack of Clear Communication and Notification**: Many borrowers were not properly informed about when their payments would resume, changes in loan servicers, or the transfer of their loans. For example, some reported not receiving notices about loan transfers or delinquency status, leading to unexpected late payments and credit damage.\n\n2. **Problems with Loan Servicing and Management**: Complaints highlight issues such as improper handling of accounts, errors in balances, and misapplication of payments. Some borrowers experienced unauthorized transfers, inaccurate reporting to credit bureaus, or inability to access their accounts online.\n\n3. **Interest Accumulation and Lack of Transparency**: Borrowers noted that interest continued to accrue and capitalize, often without clear explanations, making loans unmanageable over time. For example, some saw their balances balloon due to compounde

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [53]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [54]:
semantic_documents = semantic_chunker.split_documents(loan_complaint_data[:20])

Let's create a new vector store.

In [55]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Loan_Complaint_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [56]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [57]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [58]:
semantic_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided complaints, appears to be problems related to loan servicing and administration. This includes issues such as:\n\n- Struggling to repay loans due to administrative problems or confusing documentation\n- Errors or discrepancies in reported account status, such as loans being incorrectly marked as delinquent or in default\n- Lack of transparency and communication from loan servicers\n- Problems with payment processing, auto-debit setups, or incorrect payment amounts\n- Issues with loan forgiveness, cancellation, or discharge processes\n- Unauthorized or improper reporting of loan information to credit bureaus\n- Disputes over loan amounts, account status, or legal breaches related to data privacy\n\nOverall, many complaints revolve around errors, mismanagement, or lack of clarity in loan servicing, which causes borrower stress and financial complications.'

In [59]:
semantic_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, several complaints indicate that they were handled in a timely manner, with responses marked as "Yes" for timely response and "Closed with explanation." However, there is at least one complaint where the consumer reports that despite multiple communications and attempts to resolve issues, the company did not respond to their written complaints or questions, indicating that some complaints may not have been handled promptly or adequately.\n\nSpecifically, the complaint about Nelnet (Complaint ID: 13331376) describes that even after sending multiple certified letters detailing misconduct and violations of law, Nelnet never responded to the complaint, nor provided answers to the raised questions. The company\'s response was "Closed with explanation," which suggests that the complaint was not adequately addressed.\n\n**Conclusion:**  \nYes, some complaints did not get handled in a timely manner or did not receive proper response, as evidenced by the compl

In [60]:
semantic_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to issues such as receiving bad or unclear information from their lenders or servicers, difficulties in verifying or documenting their loans, and disputes over the legitimacy or status of their debts. Some borrowers experienced delays or errors in payment processing, leading to unintended default or missed payments. Additionally, in certain cases, borrowers faced challenges related to data breaches or improper reporting, which affected their credit reports and ability to manage repayment. Overall, lack of transparency, administrative errors, and legal disputes contributed to the failure to repay loans.'

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

✅ Answer:

When sentences are short and repetitive, like in FAQs, semantic chunking may create too many small or similar chunks.
This happens because the model detects high similarity between nearly all sentences.
As a result, chunks become less meaningful and harder to retrieve effectively.
To fix this, increase the breakpoint_threshold to reduce sensitivity.
You can also change breakpoint_threshold_type to "standard_deviation" for more adaptive behavior.
Setting a higher min_chunk_size can help form more useful chunks.
In some cases, switching to rule-based or fixed-size chunking works better.

# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

In [61]:
# Create a sample.txt file with the following content:
with open("sample.txt", "w") as f:
    f.write("""
    Dengue fever is a mosquito-borne tropical disease caused by the dengue virus.
    Symptoms typically begin three to fourteen days after infection. They include high fever,
    headache, vomiting, muscle and joint pains, and a characteristic skin rash.
    In severe cases, it can lead to bleeding, low platelet count, and blood plasma leakage.
    There is no specific antiviral treatment for dengue; supportive care is recommended.
    """)

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [89]:
# ✅ Set your OpenAI API key
os.environ["OPENAI_API_KEY"] = "your-openai-api-key-here"  # Replace with your key


In [13]:
# ✅ Clean install from GitHub - the working version
# !pip uninstall -y ragas
# !pip install git+https://github.com/explodinggradients/ragas.git@main

# ✅ STEP 1: Imports
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI

from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    AnswerRelevancy,
    Faithfulness,
    ContextPrecision,
    ContextRecall
)

import pandas as pd
import os

# ✅ Set your OpenAI API key
os.environ["OPENAI_API_KEY"] = "your-openai-api-key-here"  # Replace with your key

# ✅ STEP 2: Write sample text file
sample_text = """
Dengue fever is a mosquito-borne tropical disease caused by the dengue virus.
Symptoms typically begin three to fourteen days after infection. They include high fever,
headache, vomiting, muscle and joint pains, and a characteristic skin rash.
In severe cases, it can lead to bleeding, low platelet count, and blood plasma leakage.
There is no specific antiviral treatment for dengue; supportive care is recommended.
"""
with open("sample.txt", "w") as f:
    f.write(sample_text)

# ✅ STEP 3: Load and split documents
docs = TextLoader("sample.txt").load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = splitter.split_documents(docs)

# ✅ STEP 4: Create golden dataset (Q/A/Context aligned)
contexts = [
    "Dengue fever is a mosquito-borne tropical disease caused by the dengue virus.",
    "Symptoms typically begin three to fourteen days after infection.",
    "They include high fever, headache, vomiting, muscle and joint pains, and a characteristic skin rash.",
    "In severe cases, it can lead to bleeding, low platelet count, and blood plasma leakage.",
    "There is no specific antiviral treatment for dengue; supportive care is recommended."
]

questions = [
    "What causes dengue fever?",
    "How soon do symptoms appear?",
    "What are common symptoms of dengue?",
    "What complications can dengue cause?",
    "Is there any antiviral treatment for dengue?"
]

answers = [
    "dengue virus",
    "three to fourteen days",
    "high fever, headache, vomiting, muscle and joint pains, and a characteristic skin rash",
    "bleeding, low platelet count, and blood plasma leakage",
    "no specific antiviral treatment"
]

golden_dataset = Dataset.from_dict({
    "question": questions,
    "reference": answers,  # Rename 'answer' to 'reference' for Ragas compatibility
    "context": contexts
})

# ✅ STEP 5: Build vectorstore retriever
embedding = OpenAIEmbeddings()
faiss_vectorstore = FAISS.from_documents(splits, embedding)
retriever = faiss_vectorstore.as_retriever()

# ✅ STEP 6: Create QA chain
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)
qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)

# ✅ STEP 7: Instantiate metrics and evaluator LLM
from ragas.llms import LangchainLLMWrapper

# Metric objects
metrics = [
    AnswerRelevancy(),
    Faithfulness(),
    ContextPrecision(),
    ContextRecall()
]

# Create evaluator LLM wrapper
evaluator_llm = LangchainLLMWrapper(llm)

# ✅ STEP 8: Run the QA chain on our dataset to get responses and contexts
responses = []
retrieved_contexts = []

for question in golden_dataset["question"]:
    response = qa_chain.invoke({"query": question})
    responses.append(response["result"])
    
    # Get the retrieved documents from the retriever
    retrieved_docs = retriever.get_relevant_documents(question)
    retrieved_contexts.append([doc.page_content for doc in retrieved_docs])

# Add responses and contexts to the dataset
golden_dataset = golden_dataset.add_column("response", responses)
golden_dataset = golden_dataset.add_column("retrieved_contexts", retrieved_contexts)

# ✅ STEP 9: Evaluate using Ragas
results = evaluate(
    dataset=golden_dataset,
    metrics=metrics,
    llm=evaluator_llm
)

# ✅ STEP 9: Display evaluation results
results_df = results.to_pandas()
print(results_df) 

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

                                     user_input  \
0                     What causes dengue fever?   
1                  How soon do symptoms appear?   
2           What are common symptoms of dengue?   
3          What complications can dengue cause?   
4  Is there any antiviral treatment for dengue?   

                                  retrieved_contexts  \
0  [Dengue fever is a mosquito-borne tropical dis...   
1  [Dengue fever is a mosquito-borne tropical dis...   
2  [Dengue fever is a mosquito-borne tropical dis...   
3  [Dengue fever is a mosquito-borne tropical dis...   
4  [Dengue fever is a mosquito-borne tropical dis...   

                                            response  \
0  Dengue fever is caused by the dengue virus, wh...   
1  Symptoms of dengue fever typically begin three...   
2  Common symptoms of dengue fever include high f...   
3  Dengue fever can lead to complications such as...   
4  There is no specific antiviral treatment for d...   

                   